PROJETO: Análise de vendas automotivas<br>
ETAPA: Load Dos Dados<br>
ORIGEM: Dataset/Tratados<br>
DESTINO INICIAL: Bigquery/automotive_staging

#### conexão com BigQuery

In [1]:
import pandas as pd
from google.cloud import bigquery
from pathlib import Path


In [2]:
# Configuração do projeto
PROJECT_ID = "machados-vikings"

# Criando cliente BigQuery
client = bigquery.Client(
    project=PROJECT_ID
)

print("Conexão com BigQuery estabelecida!")
print(f"Projeto: {client.project}")


Conexão com BigQuery estabelecida!
Projeto: machados-vikings


#### Teste os Dataset

In [3]:
DATASET_ID = "automotive_staging"

dataset = client.get_dataset(
    f"{PROJECT_ID}.{DATASET_ID}"
)

print("Dataset encontrado!")
print(f"Dataset: {dataset.dataset_id}")
print(f"Localização: {dataset.location}")

Dataset encontrado!
Dataset: automotive_staging
Localização: us-west1


In [4]:
# Configurações
PROJECT_ID = "machados-vikings"
DATASET_ID = "automotive_staging"
LOCATION = "us-west1"

# Cliente BigQuery
client = bigquery.Client(project=PROJECT_ID)

# Diretório dos arquivos
BASE_DIR = Path.cwd().parent
pasta_csv = BASE_DIR / "Dataset" / "tratados"

# Configuração da carga
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)


In [ ]:
# Realizando a carga dos arquivos CSV tratados para o BigQuery,
# criando as tabelas na camada Staging.
resultados = []

for arquivo in sorted(pasta_csv.glob("*.csv")):

    nome_tabela = arquivo.stem

    table_id = (
        f"{PROJECT_ID}."
        f"{DATASET_ID}."
        f"stg_{nome_tabela}"
    )
    print(f"Iniciando LOAD: {nome_tabela}")

    with arquivo.open("rb") as csv_file:

        job = client.load_table_from_file(
            csv_file,
            table_id,
            job_config=job_config,
            location=LOCATION
        )
    job.result()
    resultados.append({
        "tabela": nome_tabela,
        "registros": job.output_rows,
        "status": job.state
    })
print("Carga de todas as tabelas finalizada!")

Iniciando LOAD: customers
LOAD concluído: customers
Registros: 25109
----------------------------------------
Iniciando LOAD: funnel
LOAD concluído: funnel
Registros: 30580
----------------------------------------
Iniciando LOAD: ibge_genders
LOAD concluído: ibge_genders
Registros: 100787
----------------------------------------
Iniciando LOAD: products
LOAD concluído: products
Registros: 333
----------------------------------------
Iniciando LOAD: regions
LOAD concluído: regions
Registros: 5570
----------------------------------------
Iniciando LOAD: stores
LOAD concluído: stores
Registros: 3026
----------------------------------------
Carga de todas as tabelas finalizada!
